In [1]:
import warnings
warnings.filterwarnings("ignore")

from rfdetr import RFDETRLarge
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import os

In [ ]:
base_dir = Path("runs/rfdetr_large")
base_dir.mkdir(parents=True, exist_ok=True)

ds_dir = Path(f"runs/rfdetr_large/run{len(os.listdir(base_dir)) + 1}")
ds_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
model = RFDETRLarge(
    resolution=336,
    gradient_checkpointing=True
)

model.train(
    dataset_dir=r"data/rfdetr",
    epochs=100,
    batch_size=4,
    grad_accum_steps=8,

    lr=1e-4,
    lr_encoder=1.5e-4,
    weight_decay=1e-4,
    lr_scheduler="cosine",

    use_ema=True,

    early_stopping=True,
    early_stopping_patience=5,
    early_stopping_min_delta= 1e-3,
    skip_best_epochs=5,

    eval_interval=1,
    log_per_class_metrics=True,
    seed=42,

    output_dir= ds_dir
)

Val (ema) (Epoch 28/100) — Overall Metrics           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃           mAP            ┃  mAR   ┃         F1 sweep         ┃
┡━━━━━━━━┯━━━━━━━━┯━━━━━━━━╇━━━━━━━━╇━━━━━━━━┯━━━━━━━━┯━━━━━━━━┩
│ 50:95  │   50   │   75   │  @500  │   F1   │  Prec  │ Recall │
├────────┼────────┼────────┼────────┼────────┼────────┼────────┤
│ 0.4867 │ 0.7074 │ 0.5266 │ 0.6979 │ 0.6879 │ 0.7332 │ 0.6509 │
└────────┴────────┴────────┴────────┴────────┴────────┴────────┘
          Val (ema) (Epoch 28/100) — Per-class Metrics          
┏━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ Class      ┃ AP 50:95 ┃     AR ┃     F1 ┃ Precision ┃ Recall ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ Car        │   0.6149 │ 0.6964 │ 0.8315 │    0.8192 │ 0.8443 │
│ Motorcycle │   0.3729 │ 0.4726 │ 0.7685 │    0.7866 │ 0.7512 │
│ Bus        │   0.6252 │ 0.7713 │ 0.7692 │    0.7979 │ 0.7426 │
│ Truck      │   0.4990 │ 0.7682 │ 0.6253 │    0.6802 │ 0.5785 │
│ Tuktuk     │   0.3771 │ 0.6184 │ 0.6136 │    0.6923 │ 0.5510 │
│ Van        │   0.4352 │ 0.7929 │ 0.5224 │    0.5469 │ 0.5000 │
│ Pickup     │   0.4286 │ 0.7394 │ 0.5622 │    0.6047 │ 0.5253 │
│ Songthaew  │   0.5404 │ 0.7238 │ 0.8108 │    0.9375 │ 0.7143 │
└────────────┴──────────┴────────┴────────┴───────────┴────────┘

Monitored metric __rfdetr_effective_map__ did not improve in the last 5 records. Best score: 0.489. Signaling Trainer to stop.


[2026-09-28 23:48:40] [INFO] rf-detr - Best total checkpoint saved from EMA (regular=0.0000, ema=0.4899)


In [9]:
NUM_CLASSES = 8

model = RFDETRLarge(
    pretrain_weights= ds_dir / "checkpoint_best_ema.pth"
)

image_dir = Path(r"data/raw/test/test")

rows = []
row_id = 0

for image_path in tqdm(list(image_dir.glob("*")), desc="Predicting"):

    detections = model.predict(
        str(image_path),
        threshold=0.01
    )

    for box, conf, cls_id in zip(
        detections.xyxy,
        detections.confidence,
        detections.class_id
    ):
        cls_id = int(cls_id)

        # Skip background / invalid class
        if not 0 <= cls_id < NUM_CLASSES:
            continue

        x1, y1, x2, y2 = map(float, box)

        # Skip degenerate boxes
        if x2 <= x1 or y2 <= y1:
            continue

        rows.append({
            "id": row_id,
            "image_id": image_path.name,
            "class_id": cls_id,
            "confidence": float(conf),
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
        })

        row_id += 1

df_pred = pd.DataFrame(
    rows,
    columns=[
        "id",
        "image_id",
        "class_id",
        "confidence",
        "x1",
        "y1",
        "x2",
        "y2",
    ]
)

# Final validation
assert df_pred["class_id"].between(0, 7).all()
assert (df_pred["x2"] > df_pred["x1"]).all()
assert (df_pred["y2"] > df_pred["y1"]).all()

df_pred.to_csv(
    ds_dir / "submission.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Predictions:", len(df_pred))

[2026-09-29 00:39:06] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-29 00:39:06] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-29 00:39:06] [WARNING] rf-detr - Checkpoint has 8 classes but model is configured for 90. Using checkpoint class count (8). Pass num_classes=8 to suppress this warning.
Predicting: 100%|██████████| 1013/1013 [00:41<00:00, 24.48it/s]


Predictions: 272962
